# Project 58 — Local Calendar Planner Agent

**Propose schedules and manage a local calendar state with LangGraph and Ollama.**

| Component | Choice |
|-----------|--------|
| LLM runtime | Ollama (qwen3:8b) |
| Framework | LangGraph |
| Calendar | In-memory Python state |
| Interface | Jupyter notebook |

## 1 · What You Will Learn

1. How to build a **scheduling agent** that understands time constraints
2. How to model **calendar state** and detect conflicts
3. How to use an LLM to **propose optimal meeting times**
4. How to implement **LangGraph** with a mocked calendar backend
5. How to handle **rescheduling** and priority-based scheduling

## 2 · Why This Project Matters

Scheduling is a common AI assistant task. The challenge isn't just finding free slots —
it's understanding priorities, preferences, and constraints. This project teaches you
how to combine state management with LLM reasoning for practical planning tasks.

## 3 · Environment Setup

**Prerequisites:**
- Ollama running with `qwen3:8b`

In [ ]:
# !pip install -q langchain langchain-ollama langgraph

## 4 · Imports and Configuration

In [ ]:
import json
from datetime import datetime, timedelta
from typing import TypedDict

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.1)
print(f'LLM ready: {MODEL}')

## 5 · Build the Calendar Backend

A simple in-memory calendar with conflict detection. In production, this would
connect to Google Calendar, Outlook, or a local CalDAV server.

In [ ]:
class LocalCalendar:
    """Simple in-memory calendar with conflict detection."""

    def __init__(self):
        self.events = []

    def add_event(self, title: str, start: datetime, end: datetime,
                  priority: str = 'normal') -> dict:
        event = {
            'id': len(self.events) + 1,
            'title': title,
            'start': start.isoformat(),
            'end': end.isoformat(),
            'priority': priority,
            'duration_min': int((end - start).total_seconds() / 60),
        }
        conflicts = self.find_conflicts(start, end)
        if conflicts:
            event['conflicts'] = [c['title'] for c in conflicts]
            event['status'] = 'conflict'
        else:
            event['status'] = 'scheduled'
            self.events.append(event)
        return event

    def find_conflicts(self, start: datetime, end: datetime) -> list:
        conflicts = []
        for ev in self.events:
            ev_start = datetime.fromisoformat(ev['start'])
            ev_end = datetime.fromisoformat(ev['end'])
            if start < ev_end and end > ev_start:
                conflicts.append(ev)
        return conflicts

    def get_free_slots(self, date: datetime, slot_duration_min: int = 60) -> list:
        """Find free time slots on a given date."""
        day_start = date.replace(hour=9, minute=0, second=0)
        day_end = date.replace(hour=17, minute=0, second=0)

        # Get events for this day
        day_events = []
        for ev in self.events:
            ev_start = datetime.fromisoformat(ev['start'])
            if ev_start.date() == date.date():
                day_events.append(ev)
        day_events.sort(key=lambda e: e['start'])

        # Find gaps
        slots = []
        current = day_start
        for ev in day_events:
            ev_start = datetime.fromisoformat(ev['start'])
            if (ev_start - current).total_seconds() >= slot_duration_min * 60:
                slots.append({'start': current.strftime('%H:%M'), 'end': ev_start.strftime('%H:%M')})
            current = max(current, datetime.fromisoformat(ev['end']))

        if (day_end - current).total_seconds() >= slot_duration_min * 60:
            slots.append({'start': current.strftime('%H:%M'), 'end': day_end.strftime('%H:%M')})

        return slots

    def get_schedule(self, date: datetime) -> str:
        """Get a formatted schedule for a date."""
        day_events = []
        for ev in self.events:
            ev_start = datetime.fromisoformat(ev['start'])
            if ev_start.date() == date.date():
                day_events.append(ev)
        day_events.sort(key=lambda e: e['start'])

        if not day_events:
            return f'No events on {date.strftime("%A, %B %d")}'

        lines = [f'Schedule for {date.strftime("%A, %B %d")}']
        for ev in day_events:
            s = datetime.fromisoformat(ev['start']).strftime('%H:%M')
            e = datetime.fromisoformat(ev['end']).strftime('%H:%M')
            lines.append(f'  {s}-{e}: {ev["title"]} [{ev["priority"]}]')
        return '\n'.join(lines)


# Initialize calendar with some events
cal = LocalCalendar()
base_date = datetime(2024, 4, 15)  # Monday

cal.add_event('Team Standup', base_date.replace(hour=9, minute=30),
              base_date.replace(hour=10, minute=0), 'high')
cal.add_event('Product Review', base_date.replace(hour=11, minute=0),
              base_date.replace(hour=12, minute=0), 'high')
cal.add_event('Lunch', base_date.replace(hour=12, minute=0),
              base_date.replace(hour=13, minute=0), 'normal')
cal.add_event('1:1 with Manager', base_date.replace(hour=14, minute=0),
              base_date.replace(hour=14, minute=30), 'high')

print(cal.get_schedule(base_date))
print(f'\nFree 60-min slots: {cal.get_free_slots(base_date, 60)}')

## 6 · Build the Planning Agent

The agent takes scheduling requests and uses the calendar state to propose times.

In [ ]:
class PlannerState(TypedDict):
    request: str
    current_schedule: str
    free_slots: list
    proposal: str
    result: str


def analyze_schedule(state: PlannerState) -> PlannerState:
    """Load current schedule and find free slots."""
    schedule = cal.get_schedule(base_date)
    free = cal.get_free_slots(base_date, 30)
    return {**state, 'current_schedule': schedule, 'free_slots': free}


def propose_schedule(state: PlannerState) -> PlannerState:
    """Use LLM to propose optimal scheduling."""
    prompt = ChatPromptTemplate.from_messages([
        ('system',
         'You are a scheduling assistant. Given the current calendar and free slots, '
         'propose the best time for the requested event.\n'
         'Consider:\n'
         '- Avoid back-to-back meetings when possible\n'
         '- Respect lunch breaks (12:00-13:00)\n'
         '- High-priority events get prime morning slots\n'
         '- Leave buffer time between meetings\n\n'
         'Respond with a clear proposal including the suggested time and reasoning.'),
        ('human',
         'Request: {request}\n\n'
         'Current Schedule:\n{schedule}\n\n'
         'Available Slots: {slots}'),
    ])
    chain = prompt | llm | StrOutputParser()
    proposal = chain.invoke({
        'request': state['request'],
        'schedule': state['current_schedule'],
        'slots': json.dumps(state['free_slots']),
    })
    return {**state, 'proposal': proposal}


def apply_schedule(state: PlannerState) -> PlannerState:
    """Apply the proposed schedule (simulated)."""
    result = f'Proposal generated for: {state["request"]}\n\n{state["proposal"]}'
    return {**state, 'result': result}


print('Planning functions defined.')

## 7 · Compile the LangGraph Workflow

In [ ]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(PlannerState)
workflow.add_node('analyze', analyze_schedule)
workflow.add_node('propose', propose_schedule)
workflow.add_node('apply', apply_schedule)

workflow.set_entry_point('analyze')
workflow.add_edge('analyze', 'propose')
workflow.add_edge('propose', 'apply')
workflow.add_edge('apply', END)

app = workflow.compile()
print('Calendar planner workflow ready.')

## 8 · Test Scheduling Requests

In [ ]:
def schedule_request(request: str) -> dict:
    state = app.invoke({
        'request': request,
        'current_schedule': '',
        'free_slots': [],
        'proposal': '',
        'result': '',
    })
    print(f'\n{"="*60}')
    print(f'REQUEST: {request}')
    print(f'{"="*60}')
    print(state['result'])
    return state


s1 = schedule_request('Schedule a 30-minute design review this afternoon')

In [ ]:
s2 = schedule_request('Find time for a 1-hour deep work block with no interruptions')

In [ ]:
s3 = schedule_request('Schedule a 45-minute interview with a candidate, high priority')

In [ ]:
s4 = schedule_request('I need 15 minutes for a quick sync with the sales team')

## 9 · Weekly Overview

Generate a weekly planning summary.

In [ ]:
# Add events for the rest of the week
for day_offset in range(1, 5):
    day = base_date + timedelta(days=day_offset)
    cal.add_event('Team Standup', day.replace(hour=9, minute=30),
                  day.replace(hour=10, minute=0), 'high')

# Tuesday extras
tue = base_date + timedelta(days=1)
cal.add_event('Sprint Planning', tue.replace(hour=10, minute=0),
              tue.replace(hour=11, minute=30), 'high')

# Wednesday extras
wed = base_date + timedelta(days=2)
cal.add_event('All Hands', wed.replace(hour=15, minute=0),
              wed.replace(hour=16, minute=0), 'high')

# Print week schedule
print('WEEKLY OVERVIEW')
print('=' * 50)
for day_offset in range(5):
    day = base_date + timedelta(days=day_offset)
    print(f'\n{cal.get_schedule(day)}')
    free = cal.get_free_slots(day, 30)
    print(f'  Free 30-min slots: {len(free)}')

## 10 · LLM Weekly Planning Advice

In [ ]:
week_data = []
for day_offset in range(5):
    day = base_date + timedelta(days=day_offset)
    week_data.append(cal.get_schedule(day))

advice_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a productivity coach. Given a weekly schedule, provide:\n'
     '1. Observations about meeting density\n'
     '2. Best days/times for deep work\n'
     '3. Scheduling improvement suggestions\n'
     'Be practical and specific.'),
    ('human', 'Weekly schedule:\n{schedule}'),
])
chain = advice_prompt | llm | StrOutputParser()
advice = chain.invoke({'schedule': '\n\n'.join(week_data)})

print('WEEKLY PLANNING ADVICE')
print('=' * 40)
print(advice)

## 11 · Save Results

In [ ]:
output = {
    'scheduling_requests': [
        {'request': s1['request'], 'proposal': s1['proposal']},
        {'request': s2['request'], 'proposal': s2['proposal']},
        {'request': s3['request'], 'proposal': s3['proposal']},
        {'request': s4['request'], 'proposal': s4['proposal']},
    ],
    'weekly_advice': advice,
}
with open('calendar_results.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print('Results saved.')

## 12 · Limitations

- **Mocked calendar** — not connected to a real calendar service
- **Single-day focus** — doesn't optimize across multiple days
- **No attendee management** — doesn't check other people's availability
- **No timezone handling** — assumes single timezone
- The LLM proposes times but doesn't parse them back into structured events

## 13 · How to Improve

1. **Connect to real calendar** via Google Calendar API or CalDAV
2. **Multi-attendee scheduling**: check availability across participants
3. **Recurring events**: handle daily/weekly recurring meetings
4. **Natural language time parsing**: 'next Tuesday afternoon' → datetime
5. **Preference learning**: remember user scheduling preferences

## 14 · Mini Challenge

1. Add a `reschedule_event` function that moves an existing event
2. Implement multi-day scheduling (find the best day AND time)
3. Add meeting type preferences (e.g., interviews in the morning, reviews afternoon)

## 15 · Key Takeaways

| Concept | What You Practiced |
|---------|-------------------|
| Calendar modeling | In-memory state with conflict detection |
| LLM planning | AI proposes times based on constraints |
| LangGraph workflow | Analyze → Propose → Apply pipeline |
| Weekly planning | Multi-day schedule optimization |
| Local-first | Ollama + Python calendar state |